# Anggota B — Tahap 2: Baseline Modeling & Pemilihan Normalisasi (`none` vs `peak`)

Notebook ini membandingkan performa baseline **semua 90 fitur** untuk dua versi normalisasi (`features_none.csv` vs `features_peak.csv`) menggunakan **SVM** dan **Random Forest (RF)**.

**Aturan anti-kebocoran:**
- Tuning hyperparameter menggunakan `GroupKFold(n_splits=4)` berdasarkan `actor` pada **train (aktor 1–16)** saja.
- `StandardScaler` diletakkan di dalam `Pipeline` dan hanya di-fit pada train.
- Pemilihan versi normalisasi ditentukan dari **macro F1 pada validation (aktor 17–20)**.
- Data **test (aktor 21–24)** tidak disentuh sama sekali.


In [ ]:
import json, os, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/project07'
    OUT_DIR = BASE_DIR
except ImportError:
    BASE_DIR = r'E:\College\UB\S2\AI\Projek akhir\AI-Projek'
    OUT_DIR = r'E:\College\UB\S2\AI\Projek akhir\AI-Projek'

sys.path.insert(0, os.path.join(BASE_DIR, 'src'))
sys.path.insert(0, os.path.join(OUT_DIR, 'src'))

import data_splits as ds
import baseline as bl


In [ ]:
fn_none = pd.read_csv(os.path.join(BASE_DIR, 'data', 'features', 'features_none.csv'))
fn_peak = pd.read_csv(os.path.join(BASE_DIR, 'data', 'features', 'features_peak.csv'))
feat_names = pd.read_csv(os.path.join(BASE_DIR, 'data', 'features', 'feature_names.csv'))['name'].tolist()

splits = ds.load_splits(BASE_DIR)
assert ds.check_split(splits) == []

# Hanya train & validation; test diabaikan
tr_none, va_none, _ = ds.split_frames(fn_none, splits)
tr_peak, va_peak, _ = ds.split_frames(fn_peak, splits)
print(f'Train: {len(tr_none)} file ({tr_none["actor"].nunique()} aktor) | Validation: {len(va_none)} file ({va_none["actor"].nunique()} aktor)')


In [ ]:
results, summary_df, estimators, selected_version, selected_model = bl.run_baseline_comparison(
    tr_none, va_none, tr_peak, va_peak, feat_names, n_jobs=-1
)
display(summary_df)
print(f'\nKeputusan normalisasi terpilih: {selected_version} (model terbaik: {selected_model})')
